# Phase Estimation of Quantum Walks
Adapted from Qualtran v0.7.0. Copyright 2023 Google LLC, Apache-2.0. Frozen commit `bcb17bb25360ed68ce3bc084645db3e929799389`.

All fifteen original cells are mapped. The source circuit display becomes an owned schedule and call graph. Source model costs, ideal input initialization and physical primitive inventories are accounted for separately. Set `FLAMEGRAPH_EXECUTABLE` to a local FlameGraph executable for the final SVG.


## Heisenberg limited phase estimation
Implements Heisenberg-Limited Phase Estimation of the Qubitized Quantum Walks as described in Section-II B. of [Encoding Electronic Spectra in Quantum Circuits with Linear T Complexity](https://arxiv.org/abs/1805.03662)

In [ ]:
import os
from collections.abc import Callable
from typing import cast

import numpy as np
from IPython import display as notebook_display

from examples.algorithms.quantum_walk_phase_estimation import (
    get_resource_state,
    hubbard_walk_models,
    ising_walk_models,
    legacy_flame_rows,
    phase_estimation_model,
)
from examples.algorithms.quantum_walk_phase_estimation import (
    phase_estimation as phase_estimation,
)
from flagquantum.compiler._component_trace.call_graph import modeled_call_graph
from flagquantum.compiler._component_trace.gate_budget import legacy_gate_budget
from flagquantum.compiler._component_trace.resource_model import (
    ComponentCost,
    estimate_component_resources,
)
from flagquantum.drawer.call_graph import export_call_graph_dot, render_call_graph_svg
from flagquantum.drawer.flame_graph import render_flame_graph_svg

SVG = cast(Callable[[bytes], object], notebook_display.SVG)
display = cast(Callable[[object], None], notebook_display.display)


### Phase estimation circuit

We start by quickly sketching the phase estimation circuit in terms of the walk operator.

In [ ]:
# The imported helpers preserve the complete original sine tensor and
# first-wire-controlled schedule. Physical bindings return an operation for
# circuit.append(phase_estimation(operation, phase=..., target=...)).
# The source notebook injects the ideal resource tensor as an input channel.
# Its circuit cost helper deliberately omits this channel and inverse QFT.


In [ ]:
num_sites = 6
eps = 1e-2
m_bits = 4
small_ising_models = ising_walk_models(num_sites, eps)
circuit = phase_estimation_model(small_ising_models, m_bits)
small_ising_circuit = circuit
resource_state = get_resource_state(m_bits)
schedule: list[tuple[str, int | None, int]] = [("controlled walk", 0, 1)]
for i in range(1, m_bits):
    schedule.extend(
        (("reflection(0)", i, 1), ("walk", None, 2**i), ("reflection(0)", i, 1))
    )
print(schedule)
graph = modeled_call_graph(circuit, max_depth=1)
display(
    SVG(
        render_call_graph_svg(
            export_call_graph_dot(
                graph.root, graph.direct_calls, title=lambda node: node.name
            )
        )
    )
)


### Computing costs

Usually, you'd define a Bloq that captures the entire phase estimation circuit, but we can use a little helper function to compute the gate counts directly from the bloqs encountered within the Cirq circuit.

In [ ]:
def get_qec_gates_cost_for_circuit(model: ComponentCost) -> dict[str, int]:
    # The explicit model has zero-cost injected PREPARE and excluded QFT nodes.
    gates = estimate_component_resources(model).gates
    return {name: value for name, value in vars(gates).items() if value}


small_ising_cost = get_qec_gates_cost_for_circuit(circuit)
display(small_ising_cost)


## Resource estimates for 1D Ising model using generic SELECT / PREPARE 

In [ ]:
num_sites = 200
eps = 1e-5
m_bits = 14
large_ising_models = ising_walk_models(num_sites, eps)
circuit = phase_estimation_model(large_ising_models, m_bits)
large_ising_cost = get_qec_gates_cost_for_circuit(circuit)
print(large_ising_cost)


## Resource estimates for 2D Hubbard model using specialized SELECT / PREPARE 
Phase estimation of walk operator for 2D Hubbard Model using SELECT and PREPARE circuits from Section V of https://arxiv.org/abs/1805.03662

In [ ]:
x_dim, y_dim = 20, 20
t = 20
mu = 4 * t
N = x_dim * y_dim * 2
qlambda = 2 * N * t + (N * mu) // 2
delta_e = t / 100
m_bits = int(np.ceil(np.log2(qlambda * np.pi * np.sqrt(2) / delta_e)))
large_hubbard_models = hubbard_walk_models(x_dim, t, mu)
circuit = phase_estimation_model(large_hubbard_models, m_bits)
large_hubbard_cost = get_qec_gates_cost_for_circuit(circuit)
print(large_hubbard_cost)


In [ ]:
# The complete included QPE additionally prices LP preparation and QFT.
qpe = phase_estimation_model(large_hubbard_models, m_bits, include_window_and_qft=True)
full_qpe_cost = get_qec_gates_cost_for_circuit(qpe)
print(full_qpe_cost)


## Flame Graphs to visualize cost for QPE on Qubitized walk operator for 2D Hubbard model

In [ ]:
qpe_small = phase_estimation_model(
    hubbard_walk_models(2, 2, 8), 14, include_window_and_qft=True
)
small_qpe_legacy = legacy_gate_budget(estimate_component_resources(qpe_small).gates)
print(small_qpe_legacy)
flame_rows = legacy_flame_rows(qpe_small)
assert sum(int(row.rsplit("\t", 1)[1]) for row in flame_rows) == small_qpe_legacy["t"]
# Supply a local FlameGraph executable, as required by the shared renderer.
flame_svg = render_flame_graph_svg(
    flame_rows, executable=os.environ.get("FLAMEGRAPH_EXECUTABLE", "flamegraph.pl")
)
display(SVG(flame_svg))


## Physical and model boundaries

The notebook-specific schedule controls the first phase wire and applies QFT in the original word order. The shared QPE class normally starts at the final wire. The owned binding helper reverses only the scheduling view and restores QFT ordering, with an explicit identity PREPARE boundary after the caller injects the ideal sine tensor. It remains an operation appended with `circuit.append(op)`. The source StatePreparationChannel represents ideal initialization, whose physical synthesis is excluded.

The original cost helper omits resource-state preparation and QFT. The separate full-QPE query includes both. The flame graph uses the original historical T budget, including seven T per CSwap, and retains rotation and Clifford totals separately. These are model estimates. Large Ising/Hubbard primitive lowering, amplitudes and workspace cleanup remain under Q-0631 admission gaps. Small executable branches reuse the source-file acceptance and notebook-specific binding tests.

The untouched source reports 9,167,472 historical Clifford gates for the small Hubbard factory, while its own QEC-to-historical conversion reports 8,314,766. The native conversion matches the latter. Both give 3,542,140 T gates and 131,428 rotations. This upstream accounting difference is retained explicitly.
